# EfficientAD and Patchcore implementation

## Setup
Before launching the training and inference scripts it's necessary to clone the project repository and to update the paths configuration inserting the right paths for input and output.

In [ ]:
!git clone https://github.com/emanuelepietrocometti/anomaly_detection_for_textile_industry.git
%cd anomaly_detection_for_textile_industry

In [ ]:
!pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130

In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Tesi/MVTec"
COLAB_DESTINATION = "/content/anomaly_detection_for_textile_industry/data/mvtec"

os.makedirs(COLAB_DESTINATION, exist_ok=True)

!cp -rp "{DATASET_PATH}"/* "{COLAB_DESTINATION}"

## Training
These cells train **EfficientAD** and **PatchCore** from scratch (on the fly) and produce the final exported model in TorchScript/PyTorch format (`.pt`). ONNX is **not** produced here — convert the `.pt` to ONNX externally afterwards.

Because `config.yaml` uses a single, model-specific output-paths block, the helper `configure_paths_for(<baseline>)` rewrites that block before each run so the two models write to separate `results/` and `exports/` folders instead of overwriting each other. `sync_to_drive(<baseline>)` then copies **both** the results and the exported `.pt` model back to Google Drive.

Run order: define the helpers, then train + sync each model.

In [ ]:
import os
import re
import shutil

REPO = "/content/anomaly_detection_for_textile_industry"
DRIVE_ROOT = "/content/drive/MyDrive/Tesi/results"

# Model-specific output naming used inside config.yaml and the exports/ folder.
MODEL_PATHS = {
    "efficientad": {"results_dir": "EfficientAd", "pt": "efficientad_pt"},
    "patchcore":   {"results_dir": "Patchcore",   "pt": "patchcore_pt"},
}


def configure_paths_for(baseline, config_path="config.yaml"):
    """Rewrite ONLY the model-specific keys of the `paths:` block in config.yaml
    so the selected baseline writes to its own results/ and exports/ folders.
    Comments and every other key are left untouched."""
    p = MODEL_PATHS[baseline]
    rd = p["results_dir"]
    updates = {
        "symlink_path":           f'"results/{rd}/textiles_dataset/latest"',
        "anomaly_images":         f'"results/{rd}/anomaly_images"',
        "report_path":            f'"results/{rd}/report"',
        "auroc_path":             f'"results/{rd}/AUROC"',
        "checkpoint_destination": f'"checkpoints/{rd}-tested.ckpt"',
        "exports_pt_path":        f'"exports/{p["pt"]}"',
        "config_dst_path":        f'"results/{rd}/config"',
    }
    with open(config_path) as f:
        lines = f.readlines()

    in_paths = False
    for i, line in enumerate(lines):
        if re.match(r'^paths:\s*$', line):
            in_paths = True
            continue
        if in_paths:
            # A new non-indented, non-comment line ends the paths section.
            if re.match(r'^\S', line) and not line.startswith('#'):
                in_paths = False
                continue
            m = re.match(r'^(\s*)([A-Za-z_]+):\s', line)
            if m and m.group(2) in updates:
                lines[i] = f'{m.group(1)}{m.group(2)}: {updates[m.group(2)]}\n'

    with open(config_path, "w") as f:
        f.writelines(lines)
    print(f"[CONFIG] paths section switched to '{baseline}' (results/{rd}, exports/{p['pt']})")


def _copy_tree(src, dst):
    if not os.path.exists(src):
        print(f"[SKIP] not found: {src}")
        return
    os.makedirs(dst, exist_ok=True)
    for item in os.listdir(src):
        s, d = os.path.join(src, item), os.path.join(dst, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, symlinks=True, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"[OK] {src} -> {dst}")


def sync_to_drive(baseline, drive_root=DRIVE_ROOT):
    """Copy the results folder AND the exported .pt model of `baseline` to Drive.
    (ONNX is produced externally from the .pt, outside this repository.)"""
    p = MODEL_PATHS[baseline]
    label = p["results_dir"]
    _copy_tree(f"{REPO}/results/{label}", f"{drive_root}/{label}/results")
    _copy_tree(f"{REPO}/exports/{p['pt']}", f"{drive_root}/{label}/exports/pt")
    print(f"[DONE] '{baseline}' results + exported .pt model synced to Drive.")


### EfficientAD
Train EfficientAD from scratch and export its final model.

In [ ]:
# --- EfficientAD: train from scratch and export the final model ---
%cd /content/anomaly_detection_for_textile_industry

configure_paths_for("efficientad")

# No --timestamp: a fresh timestamp is generated so training starts from scratch.
!python main.py --baseline efficientad --timestamp 10082026 --mode unsupervised

In [ ]:
# Copy EfficientAD results AND the exported .pt model to Google Drive
sync_to_drive("efficientad")

### PatchCore
Train PatchCore from scratch and export its final model. `configure_paths_for("patchcore")` redirects the output paths so PatchCore does not overwrite the EfficientAD results.

In [ ]:
# --- PatchCore: train from scratch and export the final model ---
%cd /content/anomaly_detection_for_textile_industry

configure_paths_for("patchcore")

# PatchCore is a memory-bank method: keep the default (unsupervised) mode.
!python main.py --baseline patchcore --timestamp 10082026 --mode unsupervised

In [ ]:
# Copy PatchCore results AND the exported .pt model to Google Drive
sync_to_drive("patchcore")

## Hyperparameter optimization

In [ ]:
%cd /content/anomaly_detection_for_textile_industry

!python src/hyperparameter_optimization_efficientad.py

## Inference
This second script enables you to run model inference and evaluate the overall performance of the pipeline. The arguments are:
- model path: onnx model path used for the inference;
- image path: path of the image used for the inference. The program use the same image for all the batches. This is done beacase the goal of this analysis is only the inference time;
- iterations: number of iterations for each batch;
- device: allow to chose the acceleration device between 'cpu', 'cuda' and 'tensorrt'.

In [ ]:
%cd /content/anomaly_detection_for_textile_industry

# inference.py runs on an ONNX model. This repo no longer exports ONNX:
# convert the trained .pt (exports/<model>_pt/...) to ONNX externally first,
# then point --model to that .onnx file.
!python inference.py --model <model_path> --image <image_path> --iterations 1000 --device tensorrt